In [8]:
import cv2
import mediapipe as mp
import numpy as np
import math
import os

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# -----------------------------------
# VOLUME CONTROL
# -----------------------------------

from pycaw.pycaw import AudioUtilities

# Get default speaker device
device = AudioUtilities.GetSpeakers()

# Get endpoint volume object
volume = device.EndpointVolume

def set_volume(distance):

    # Convert finger distance → volume %
    volume_percent = np.interp(
        distance,
        [20, 200],
        [0, 100]
    )

    # Set system volume
    volume.SetMasterVolumeLevelScalar(
        volume_percent / 100,
        None
    )

# -----------------------------------
# BRIGHTNESS CONTROL
# -----------------------------------

def set_brightness(distance):

    brightness = int(
        np.interp(
            distance,
            [20, 200],
            [0, 100]
        )
    )

    brightness = max(0, min(100, brightness))

    os.system(
        f'powershell (Get-WmiObject -Namespace root/WMI -Class WmiMonitorBrightnessMethods).WmiSetBrightness(1,{brightness})'
    )

# -----------------------------------
# LOAD MEDIAPIPE MODEL
# -----------------------------------

model_path = "hand_landmarker.task"

base_options = python.BaseOptions(
    model_asset_path=model_path
)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2
)

detector = vision.HandLandmarker.create_from_options(
    options
)

# -----------------------------------
# SMOOTHING VARIABLES
# -----------------------------------

smooth_distance_right = 0
smooth_distance_left = 0

# Smoothing strength
alpha = 0.2

# -----------------------------------
# DEAD ZONE SETTINGS
# -----------------------------------

dead_zone = 5

last_right_distance = 0
last_left_distance = 0

# -----------------------------------
# OPEN CAMERA
# -----------------------------------

cap = cv2.VideoCapture(0)

while True:

    success, frame = cap.read()

    if not success:
        break

    # Mirror webcam
    frame = cv2.flip(frame, 1)

    # Convert BGR → RGB
    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    # Convert to MediaPipe image
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    # Detect hands
    detection_result = detector.detect(
        mp_image
    )

    # -----------------------------------
    # PROCESS HANDS
    # -----------------------------------

    if detection_result.hand_landmarks:

        for idx, hand_landmarks in enumerate(
            detection_result.hand_landmarks
        ):

            h, w, c = frame.shape

            # Detect hand type
            detected_hand = detection_result.handedness[idx][0].category_name

            # Fix mirrored webcam labels
            if detected_hand == "Left":
                hand_type = "Right"
            else:
                hand_type = "Left"

            # Thumb tip
            thumb_tip = hand_landmarks[4]

            # Index tip
            index_tip = hand_landmarks[8]

            x1 = int(thumb_tip.x * w)
            y1 = int(thumb_tip.y * h)

            x2 = int(index_tip.x * w)
            y2 = int(index_tip.y * h)

            # Draw thumb point
            cv2.circle(
                frame,
                (x1, y1),
                15,
                (255, 0, 0),
                -1
            )

            # Draw index point
            cv2.circle(
                frame,
                (x2, y2),
                15,
                (0, 255, 0),
                -1
            )

            # Draw line
            cv2.line(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 0, 255),
                3
            )

            # -----------------------------------
            # CALCULATE DISTANCE
            # -----------------------------------

            distance = math.hypot(
                x2 - x1,
                y2 - y1
            )

            # -----------------------------------
            # SMOOTH DISTANCE
            # -----------------------------------

            if hand_type == "Right":

                smooth_distance_right = (
                    alpha * distance
                    + (1 - alpha) * smooth_distance_right
                )

                final_distance = smooth_distance_right

            else:

                smooth_distance_left = (
                    alpha * distance
                    + (1 - alpha) * smooth_distance_left
                )

                final_distance = smooth_distance_left

            # -----------------------------------
            # CONTROL SYSTEM WITH DEAD ZONE
            # -----------------------------------

            if hand_type == "Right":

                # Ignore tiny movement
                if abs(final_distance - last_right_distance) > dead_zone:

                    set_brightness(final_distance)

                    last_right_distance = final_distance

                control_type = "Brightness"

            else:

                # Ignore tiny movement
                if abs(final_distance - last_left_distance) > dead_zone:

                    set_volume(final_distance)

                    last_left_distance = final_distance

                control_type = "Volume"

            # -----------------------------------
            # CONVERT TO PERCENT
            # -----------------------------------

            percent = int(
                np.interp(
                    final_distance,
                    [20, 200],
                    [0, 100]
                )
            )

            # Keep within limits
            percent = max(0, min(100, percent))

            # -----------------------------------
            # DISPLAY TEXT
            # -----------------------------------

            cv2.putText(
                frame,
                f"{hand_type} {control_type}: {percent}%",
                (x1, y1 - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 255),
                2
            )

    # -----------------------------------
    # SHOW WINDOW
    # -----------------------------------

    cv2.imshow(
        "Gesture Control System",
        frame
    )

    # Press q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# -----------------------------------
# CLOSE EVERYTHING
# -----------------------------------

cap.release()
cv2.destroyAllWindows()